In [1]:
#Q2. The widespread use of Large Language Models (LLMs) in academic assistance has increased concerns about citation hallucination, where models generate fabricated or inaccurate references. Design a prompt necessary to examine whether few-shot prompting can reduce citation hallucination compared to zero-shot prompting.

In [3]:
# import TF-IDF Vectorizer
# used to convert text into numerical vectors

from sklearn.feature_extraction.text import TfidfVectorizer

# Import PCA
# Used to reduce the number of features (dimensions)

from sklearn.decomposition import PCA

# Import K-Means Clustering
# Used to group similar citations together

from sklearn.ensemble import IsolationForest

import pandas as pd

#-----------------------------------------------
# step-1:-> Citation using Zero-Shot Prompt

zero_shot = [
    "Attention Is All You Need Vaswani 2017",
    "BERT Devlin 2018",
    "Neural GPT Hallucination Study Smith 2024",
    "Advanced LLM Citation Detection Brown 2025",
    "Large Transformer Analysis Johnson 2026"
]

# Step-2:->
# Citation using Few-Shot Prompt


few_shot =[
    "Attention Is All You Need Vaswani et al. NeurIPS 2017",
    "Bert Devlin et al. NAACL 2019",
    "RoBERTa Liu et al. arXiv 2019",
    "GPT-3 Brown at al. NeurIPS 2020",
    "T5 Raffel et al. JMLR 2020"]

#Step-3:-> Combine citations
citations = zero_shot+few_shot

labels = (
    ["Zero-Shot"]*len(zero_shot)
    + ["Few-Shot"]*len(few_shot)
)


# Create TF-IDF object
vectorizer = TfidfVectorizer()


# Learn the words and convert text into vectors
X = vectorizer.fit_transform(citations).toarray()


# Display TF-IDF vectors
print("TF-IDF Vectors: ")
print(X)


# Step-4:->
# Apply PCA
# Ex:-
# Before PCA -> many word features
# After PCA -> only two important components


# Keep only 2 principal components
pca = PCA(n_components=2) 


# Transform the TF-IDF vectors
X_pca = pca.fit_transform(X)


# Display reduced data
print("\n After PCA: ")
print(X_pca)


# Step-5:->
# detect suspicious citations


model = IsolationForest(
    contamination=0.30,
    random_state=42
)

# Train model and assign cluster labels
prediction = model.fit_predict(X_pca)


# Display results
results = []
print("\nCitation Analysis")



for prompt_type,citation,pred in zip(labels,citations, prediction):
    if pred == -1:
        status ="Suspicious"
    else:
        status ="Likely Genuine"
    results.append([prompt_type,citation,status])
    print(f"{prompt_type:10} | {status:35} |{citation}")


df = pd.DataFrame(
    results,
    columns=["Prompt Type","Citation","Prediction"]
)
print("\nFinal Result")
print(df)


zero_fake = sum(
    (df["Prompt Type"] == "Zero-Shot") &
    (df["Prediction"] == "Suspicious")
)

few_fake = sum(
    (df["Prompt Type"] == "Few-Shot") &
    (df["Prediction"] == "Suspicious")
)

print("\nComparision")
print("Zero-shot suspicious Citations :",zero_fake)
print("Few-shot suspicious Citations :",few_fake)

if zero_fake> few_fake:
    print("\nConclusion:")
    print("Few-shot prompting reduces citation hallucination")
elif zero_fake< few_fake:
    print("\nConclusion:")
    print("Zero-shot prompting reduces citation hallucination")
else:
    print("\nConclusion:")
    print("Both methods preformed similarly")

TF-IDF Vectors: 
[[0.37796447 0.         0.         0.         0.         0.
  0.         0.         0.         0.37796447 0.         0.
  0.         0.37796447 0.         0.         0.         0.
  0.         0.         0.         0.         0.37796447 0.
  0.         0.         0.         0.         0.         0.37796447
  0.         0.         0.         0.         0.         0.
  0.         0.         0.37796447 0.37796447]
 [0.         0.63948886 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.54362395 0.         0.         0.
  0.54362395 0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.41802399 0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.    